# STimage Kidney Benchmark (INT13/14/15/17/18/19/24 -> INT21)

Strict STimage reproduction using this repository's HEST data and gene list.
Multi-slide supervision: train on two sections, validate on a third, test on the fourth.

Reported as **STimage (feature-cached)**, not as a literal official Keras `model.fit`
run -- see `benchmark/hest_dataset.py` for the measurement that forces this and
`benchmark/STimage/README_MORPHOFM_ADAPTATION.md` for the full boundary.

Kept from upstream: the frozen ResNet50/ImageNet no-top trunk with
GlobalAveragePooling2D, the negative-binomial parameterization (upstream's own
`negative_binomial_layer`), the negative-binomial loss, Adam(1e-5) from the official
`model.compile`, and EarlyStopping on `val_loss` with upstream's `patience=10`.

Adapted data surface only: HEST whole-slide reading, spot barcode alignment,
`common_genes.txt` gene panel, fixed slide-level splits, prediction export.

In [ ]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd

# Optional OpenSlide
try:
    import openslide
    HAS_OPENSLIDE = True
except ImportError:
    HAS_OPENSLIDE = False
    print("OpenSlide not found, using PIL fallback (may be slow).")

ROOT = Path(os.environ.get("MORPHO_VC_ROOT", "../")).expanduser().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / "benchmark") not in sys.path:
    sys.path.insert(0, str(ROOT / "benchmark"))

import tensorflow as tf

from hest_dataset import (
    STIMAGE_FEATURE_DIM,
    STIMAGE_TILE_SIZE,
    bind_stimage_upstream,
    build_stimage_vectorized_head,
    extract_stimage_features,
    load_stimage_head_weights,
    save_stimage_head,
    stimage_nb_loss,
    stimage_spot_matrix,
    stimage_target_matrix,
    verify_stimage_vectorization,
)

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

# IDs (multi-slide Protocol A defaults; patched per fold by the runner)
train_ids = ['INT13', 'INT14', 'INT15', 'INT17', 'INT18', 'INT19']
val_ids = ["INT24"]
test_ids = ["INT21"]
all_ids = train_ids + val_ids + test_ids

# Paths
STIMAGE_ROOT = ROOT / "benchmark" / "STimage"
result_dir = ROOT / "benchmark" / "results" / "kidney_INT21" / "stimage"
result_dir.mkdir(parents=True, exist_ok=True)

feature_cache_dir = ROOT / "results" / "kidney_hest" / "stimage" / "features"
feature_cache_dir.mkdir(parents=True, exist_ok=True)

spatial_dir = ROOT / "data" / "spatial_data"
wsi_dir = ROOT / "data" / "hest_data" / "wsis"
common_gene_path = spatial_dir / "genesets" / "kidney_hest_v1_INT13_INT14_INT15_INT17_INT18_INT19_INT21_INT24.txt"

STIMAGE_WINDOW = STIMAGE_TILE_SIZE
STIMAGE_CROP_MODE = "tile_px"
STIMAGE_RESIZE_TO = STIMAGE_TILE_SIZE

print("Result dir:", result_dir)
print(f"Train: {train_ids} | Val: {val_ids} | Test: {test_ids}")
print("TF:", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))


In [ ]:
# ===== Helpers =====


def read_gene_list(path: Path):
    genes = path.read_text(encoding="utf-8").splitlines()
    return [g.strip() for g in genes if g.strip()]


def pearson_corr(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    m = np.isfinite(a) & np.isfinite(b)
    if m.sum() <= 1:
        return np.nan
    a, b = a[m], b[m]
    if np.all(a == a[0]) or np.all(b == b[0]):
        return np.nan
    a = a - a.mean()
    b = b - b.mean()
    denom = np.sqrt((a * a).sum()) * np.sqrt((b * b).sum())
    return float((a * b).sum() / denom) if denom > 0 else np.nan


print("Helpers ready.")


In [ ]:
# ===== Step 1: Cache the frozen ResNet50-GAP trunk features (one 2048-d vector per spot) =====

genes = read_gene_list(common_gene_path)
print(f"Genes: {len(genes)}")

# Audit gate: confirm the vectorized head and loss reproduce upstream's own
# `negative_binomial_layer` / `negative_binomial_loss` before training on them.
bind_stimage_upstream(STIMAGE_ROOT)
vectorization_check = verify_stimage_vectorization(STIMAGE_ROOT)
print("Vectorization check:",
      f"head_rel={vectorization_check['relative_head_difference']:.2e}",
      f"loss_rel={vectorization_check['relative_loss_difference_float64']:.2e}")

feature_bundle = extract_stimage_features(
    all_ids,
    hest_dir=ROOT / "data" / "hest_data",
    spatial_dir=spatial_dir,
    window=STIMAGE_WINDOW,
    crop_mode=STIMAGE_CROP_MODE,
    resize_to=STIMAGE_RESIZE_TO,
    feature_cache_dir=feature_cache_dir,
    batch_size=128,
    verbose=True,
)

for sid in all_ids:
    print(f"{sid}: features {feature_bundle[sid]['shape']}")

with open(result_dir / "vectorization_check.json", "w", encoding="utf-8") as f:
    json.dump(vectorization_check, f, indent=2)


In [ ]:
# ===== Step 2: Targets + train/valid/test matrices =====

import anndata as ad

targets = {}
for sid in all_ids:
    adata = ad.read_h5ad(spatial_dir / f"{sid}.h5ad")
    targets[sid] = stimage_target_matrix(adata, genes)
    print(f"{sid}: target {targets[sid].shape}")


def collect(slide_ids):
    feats, tgts, _ = stimage_spot_matrix(
        {s: feature_bundle[s] for s in slide_ids}, list(slide_ids), targets)
    return np.asarray(feats, dtype=np.float32), np.asarray(tgts, dtype=np.float32)


X_train, Y_train = collect(train_ids)
X_val, Y_val = collect(val_ids)
print(f"X_train {X_train.shape} | X_val {X_val.shape} | genes {len(genes)}")


In [ ]:
# ===== Step 3: Train the vectorized NB head (upstream loss / optimizer / early stopping) =====

# Adam(1e-5) is what upstream `CNN_NB_multiple_genes` compiles with; batch 32 and
# patience 20 are upstream's own benchmark settings (`Fig2/benchmarking_her2st/
# stimage_her2st.py`: 299x299 tiles, batch 32, 100 epochs, EarlyStopping patience 20).
# MAX_EPOCHS is raised above upstream's 100 on purpose. At this repository's 17,511-gene
# panel the validation loss was still falling at epoch 100 -- a 100-epoch run selected
# its best epoch at 99, i.e. the final one, meaning the cap, not convergence, ended
# training. Early stopping is upstream's own stated stopping rule, so it is allowed to
# decide here and the epoch count is only a safety bound.
LEARNING_RATE = 1.0e-5
BATCH_SIZE = 32
MAX_EPOCHS = 1000
PATIENCE = 20

head = build_stimage_vectorized_head(STIMAGE_FEATURE_DIM, len(genes), seed=SEED)
optimizer = tf.keras.optimizers.Adam(LEARNING_RATE)

def run_nb_epoch(head, optimizer, X, Y, batch_size, shuffle, seed):
    """One pass over (X, Y) using upstream STimage's objective.

    `optimizer=None` evaluates without updating, which is how validation is run.
    Loss and parameterization are upstream's; only the data layout differs (the
    trunk features are precomputed, see the notebook header).
    """
    order = np.arange(X.shape[0])
    if shuffle:
        np.random.default_rng(seed).shuffle(order)
    losses = []
    for start in range(0, len(order), batch_size):
        batch = order[start:start + batch_size]
        xb = tf.constant(X[batch])
        yb = tf.constant(Y[batch])
        if optimizer is None:
            losses.append(float(stimage_nb_loss(yb, head(xb))))
        else:
            with tf.GradientTape() as tape:
                loss = stimage_nb_loss(yb, head(xb))
            grads = tape.gradient(loss, head.trainable_variables)
            optimizer.apply_gradients(zip(grads, head.trainable_variables))
            losses.append(float(loss))
    return float(np.mean(losses)) if losses else float("nan")


def evaluate_and_save(pred_bag, true_bag, genes, out_dir):
    """Write pred_bag/true_bag/metrics.json in the shape the benchmark manifests read."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    pred_bag = np.asarray(pred_bag, dtype=np.float32)
    true_bag = np.asarray(true_bag, dtype=np.float32)
    mae = float(np.mean(np.abs(pred_bag - true_bag)))
    rmse = float(np.sqrt(np.mean((pred_bag - true_bag) ** 2)))
    corrs = np.array([pearson_corr(pred_bag[:, i], true_bag[:, i])
                      for i in range(pred_bag.shape[1])], dtype=np.float32)
    valid = np.where(np.isfinite(corrs))[0]
    mean_pearson = float(np.nanmean(corrs)) if valid.size else float("nan")
    best_idx = int(valid[np.argmax(corrs[valid])]) if valid.size else None
    best_gene = genes[best_idx] if best_idx is not None else "NA"

    print(f"MAE: {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"Mean Pearson (gene-wise): {mean_pearson:.4f}")
    print(f"Best Gene: {best_gene} (Pearson={corrs[best_idx]:.4f})" if best_idx is not None else "Best Gene: NA")

    np.save(out_dir / "pred_bag.npy", pred_bag)
    np.save(out_dir / "true_bag.npy", true_bag)
    np.save(out_dir / "gene_corr_all.npy", corrs)
    with open(out_dir / "gene_names.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(str(g) for g in genes))
    with open(out_dir / "metrics.json", "w", encoding="utf-8") as f:
        json.dump({
            "MAE": mae, "RMSE": rmse, "Mean_Pearson": mean_pearson,
            "Best_Gene": best_gene,
            "Best_Pearson": float(corrs[best_idx]) if best_idx is not None else None,
        }, f, ensure_ascii=False, indent=2)
    print("Saved pred_bag/true_bag/metrics.json ->", out_dir)
    return {"MAE": mae, "RMSE": rmse, "Mean_Pearson": mean_pearson}

best_val = float("inf")
best_epoch = -1
best_weights = None
epoch_since_best = 0
history = []

for epoch in range(MAX_EPOCHS):
    tl = run_nb_epoch(head, optimizer, X_train, Y_train, BATCH_SIZE, True, SEED + epoch)
    vl = run_nb_epoch(head, None, X_val, Y_val, BATCH_SIZE, False, SEED)
    history.append({"epoch": epoch, "train_loss": tl, "val_loss": vl})
    print(f"{epoch}, train_loss: {tl:.4f},\tvalid_loss: {vl:.4f}")

    if vl < best_val:
        best_val = vl
        best_epoch = epoch
        best_weights = [v.numpy().copy() for v in head.trainable_variables]
        epoch_since_best = 0
    else:
        epoch_since_best += 1

    if epoch_since_best >= PATIENCE:
        print(f"Early stopping at epoch {epoch + 1}")
        break

head.kernel.assign(best_weights[0])
head.bias.assign(best_weights[1])
print(f"\nBest epoch {best_epoch} | valid NB loss {best_val:.4f} (restored)")

save_stimage_head(head, result_dir / "stimage_head.npz")
pd.DataFrame(history).to_csv(result_dir / "train_log.csv", index=False)
with open(result_dir / "config.json", "w", encoding="utf-8") as f:
    json.dump({
        "method": "STimage",
        "reported_as": "STimage (feature-cached)",
        "protocol": "rotating_heldout_multi_slide",
        "train_slide_ids": train_ids, "val_slide_ids": val_ids, "test_slide_ids": test_ids,
        "learning_rate": LEARNING_RATE, "batch_size": BATCH_SIZE,
        "max_epochs": MAX_EPOCHS, "patience": PATIENCE, "seed": SEED,
        "window": STIMAGE_WINDOW, "crop_mode": STIMAGE_CROP_MODE,
        "resize_to": STIMAGE_RESIZE_TO, "feature_dim": STIMAGE_FEATURE_DIM,
        "cnn_base": "resnet50", "fine_tuning": False,
        "best_epoch": best_epoch, "best_valid_nb_loss": float(best_val),
        "n_genes": len(genes),
    }, f, ensure_ascii=False, indent=2)
print("Saved:", result_dir / "stimage_head.npz")


In [ ]:
# ===== Step 4: Evaluation + Save pred_bag/true_bag + metrics =====

# Upstream predicts the negative-binomial mean n*(1-p)/p, which is what
# `stimage/03_Prediction.py` obtains via `scipy.stats.nbinom.mean`.
test_feats, test_targets, _ = stimage_spot_matrix(
    {sid: feature_bundle[sid] for sid in test_ids}, list(test_ids), targets)
pred_bag = np.asarray(
    head.nb_mean(tf.constant(np.asarray(test_feats, dtype=np.float32))).numpy(),
    dtype=np.float32,
)
true_bag = np.asarray(test_targets, dtype=np.float32)
print("Prediction Shape:", pred_bag.shape)

evaluate_and_save(pred_bag, true_bag, genes, result_dir)


In [ ]:
# ===== Step 5: Visualization (01-style, top-4 genes on the test slide) =====

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

VIS_ID = test_ids[0]
adata_vis = ad.read_h5ad(spatial_dir / f"{VIS_ID}.h5ad")
mask_vis = adata_vis.obs["in_tissue"].to_numpy().astype(bool)
xy = adata_vis.obs[["pxl_col_in_fullres", "pxl_row_in_fullres"]].to_numpy()[mask_vis]

pred_vals = pred_bag[: mask_vis.sum()]
true_vals = true_bag[: mask_vis.sum()]
corrs_vis = np.array([pearson_corr(pred_vals[:, i], true_vals[:, i]) for i in range(pred_vals.shape[1])])
valid_idx = np.where(np.isfinite(corrs_vis))[0]
top4 = valid_idx[np.argsort(corrs_vis[valid_idx])[::-1][:4]]

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for j, gi in enumerate(top4):
    for row, (vals, tag) in enumerate([(true_vals, "measured"), (pred_vals, "predicted")]):
        ax = axes[row, j]
        sc = ax.scatter(xy[:, 0], xy[:, 1], c=vals[:, gi], s=3, cmap="viridis")
        ax.set_title(f"{genes[gi]} ({tag})\nr={corrs_vis[gi]:.3f}")
        ax.set_xticks([]); ax.set_yticks([])
        plt.colorbar(sc, ax=ax, fraction=0.046)
plt.tight_layout()
out_fig = result_dir / "top4_genes_vis.png"
plt.savefig(out_fig, dpi=150)
plt.close()
print("Saved:", out_fig)
